# Tutorial: Scoring

Audience:
- Developers computing catalyst scores and opportunity rankings.

Prerequisites:
- Access to event ingestion and OHLCV data provider.

Learning goals:
- Build catalyst signals from events and price action.
- Convert signals into per-symbol catalyst scores.
- Combine catalyst and technical inputs into opportunities.


## Outline

1. Setup imports and collect events
2. Fetch OHLCV history and build catalyst signals
3. Build catalyst score map
4. Build ranked opportunities


In [ ]:
from datetime import datetime, timedelta

from swing_screener.intelligence import (
    CatalystConfig,
    IntelligenceConfig,
    build_catalyst_score_map,
    build_catalyst_signals,
    build_opportunities,
    collect_events,
)
from swing_screener.intelligence.models import SymbolState
from swing_screener.data.providers import get_default_provider


## Step 1 - Collect recent events

Gather 72-hour event history for a small symbol basket.


In [ ]:
symbols = ["NVDA", "AMD", "INTC", "TSM", "AVGO"]
now = datetime.now()
start_dt = now - timedelta(hours=72)

events = collect_events(
    symbols=symbols,
    start_dt=start_dt,
    end_dt=now,
    provider_names=("yahoo_finance",),
)
print(f"Collected {len(events)} events")


## Step 2 - Fetch OHLCV and compute catalyst signals

Price history enables event reaction measurements used in signal filtering.


In [ ]:
provider = get_default_provider()
start_date = (start_dt - timedelta(days=30)).date().isoformat()
end_date = now.date().isoformat()
ohlcv = provider.fetch_ohlcv(symbols, start_date=start_date, end_date=end_date)

catalyst_cfg = CatalystConfig(
    lookback_hours=72,
    false_catalyst_return_z=1.5,
    min_price_reaction_atr=0.8,
    require_price_confirmation=True,
)

signals = build_catalyst_signals(events=events, ohlcv=ohlcv, cfg=catalyst_cfg, asof_dt=now)
print(f"Generated {len(signals)} signals")
for sig in signals:
    print(
        f"  {sig.symbol}: z={sig.return_z:.2f}, atr_shock={sig.atr_shock:.2f}, "
        f"false_catalyst={sig.is_false_catalyst}"
    )


## Step 3 - Build catalyst scores from valid signals

Remove false catalysts, then score by symbol with recency decay.


In [ ]:
valid_signals = [s for s in signals if not s.is_false_catalyst]
print(f"Valid signals (not false catalysts): {len(valid_signals)}")

catalyst_scores = build_catalyst_score_map(
    signals=valid_signals,
    events=events,
    themes=[],
    recency_half_life_hours=36,
)

print("Catalyst scores:")
for symbol, score in catalyst_scores.items():
    print(f"  {symbol}: {score:.3f}")


## Step 4 - Build opportunities with technical readiness

Combine technical readiness and catalyst strength using opportunity config weights.


In [ ]:
states = {s: SymbolState.new(s) for s in symbols}
technical_readiness = {
    "NVDA": 0.85,
    "AMD": 0.72,
    "INTC": 0.45,
    "TSM": 0.68,
    "AVGO": 0.75,
}

opp_cfg = IntelligenceConfig(
    opportunity__technical_weight=0.55,
    opportunity__catalyst_weight=0.45,
    opportunity__max_daily_opportunities=5,
    opportunity__min_opportunity_score=0.5,
).opportunity

opportunities = build_opportunities(
    technical_readiness=technical_readiness,
    catalyst_scores=catalyst_scores,
    symbol_states=states,
    cfg=opp_cfg,
)

print(f"Opportunities ({len(opportunities)}):")
for opp in opportunities:
    print(
        f"  {opp.symbol}: score={opp.opportunity_score:.3f}, "
        f"technical={opp.technical_readiness:.2f}, "
        f"catalyst={opp.catalyst_strength:.2f}"
    )
